# Этический паспорт ИИ-решения «КадроБот 3000»

## Назначение и границы использования модели


| Параметр | Описание |
| :--- | :--- |
| **Наименование системы** | КадроБот 3000 |
| **Назначение** | Автоматизированная предварительная оценка резюме кандидатов для ритейл-компании «МаркетПлюс» с целью ранжирования кандидатов по вероятности успешности |
| **Целевая аудитория** | Специалисты по подбору персонала (HR), рекрутеры |
| **Границы применения** | • Только для первичного скрининга резюме<br> • Не используется для финального решения о найме<br> • Не применяется для позиций с особыми требованиями к возрасту (законодательно обоснованными)<br> • Не используется для автоматического отказа без человеческого контроля |
| **Тип модели** | Градиентный бустинг / логистическая регрессия (бинарная классификация) |
| **Метрика качества** | ROC-AUC = $0.85$ (на тестовых данных) |


### Выявленные ограничения и известные биасы


*На основе технического аудита (Этап 1)*

| Тип смещения | Описание | Источник данных | Уровень риска |
| :--- | :--- | :--- | :--- |
| **Возрастная дискриминация** | Модель систематически занижает оценку кандидатам 45+ через прокси-признаки | Исторические данные найма за 5 лет | 🔴 Высокий |
| **Прокси-переменные** | `graduation_year` $(r=-0.884)$, `outdated_vocab` $(r=+0.526)$, `youth_hobby` $(r=-0.485)$, `experience` $(r=+0.913)$ коррелируют с возрастом и влияют на `model_score` | Корреляционный анализ признаков | 🔴 Высокий |
| **Историческое смещение** | Модель воспроизводит предвзятые решения прошлого: доля приглашений 45+ $= 3.8%$ vs <30 $= 94.1%$ | `historical_success` vs `model_decision` | 🔴 Высокий |
| **Ложные отказы (False Negatives)** | Успешные кандидаты 45+ получают отказ в $94.3%$ случаев | TPR(45+) $= 0.057$ vs TPR(<30) $= 0.947$ | 🔴 Высокий |


**Ключевой вывод**: Дискриминация не объясняется различиями в квалификации — при контроле `education` и `experience` разница в `model_score` между $45+$ и $<30$ составляет $-0.316$ (статистически значимо).

## Метрики справедливости


### Расчётные показатели



| Метрика | Математическая формула | Значение | Бизнес-интерпретация (на тестовой выборке) |
| :--- | :--- | :--- | :--- |
| **Демографический паритет** *(Statistical Parity)* | $\vert P(\hat{Y}=1 \vert A = \text{<30}) - P(\hat{Y}=1 \vert A = \text{45+}) \vert$ | **0.903** | **Критический перекос воронки:** Независимо от реальной квалификации, кандидат из молодой группы имеет в **24.8×** более высокий шанс получить приглашение. |
| **Равные возможности** *(Equal Opportunity)* | $\vert TPR_{\text{<30}} - TPR_{\text{45+}} \vert$ | **0.890** | **Дискриминация талантов:** Модель одобряет успешных молодых кандидатов в **16.6×** чаще, искусственно отсекая $94.3\%$ продуктивных специалистов старше 45 лет. |
| **Равная точность** *(Equalized Odds)* | $\vert FPR_{\text{<30}} - FPR_{\text{45+}} \vert + \vert TPR_{\text{<30}} - TPR_{\text{45+}} \vert$ | **1.120** | **Комплексный дисбаланс ошибок:** Модель одновременно завышает проходной балл для старшей группы и массово пропускает слабых молодых кандидатов «авансом». |
| **Справедливость через неосведомлённость** | Исключение `age` из признаков | ❌ **Неэффективно** | **Скрытая предвзятость:** Простое удаление прямой переменной возраста не решает проблему — модель полностью восстанавливает его через прокси-признаки. |


### Обоснование выбора метрик


Для задачи найма персонала выбраны **демографический паритет** и **равные возможности** по следующим причинам:
*    **Демографический паритет** (Demographic Parity):
* *    *Почему уместен*: В найме важно обеспечить равный доступ к возможности трудоустройства для всех возрастных групп, независимо от исторических диспропорций.
  *    *Философское обоснование*: Соответствует принципу **справедливости как равенства возможностей** (Ролз) — неравенство допустимо только если работает в пользу наименее преуспевающих.
*    **Равные возможности** (Equal Opportunity):
* *    *Почему уместен*: Фокусируется на справедливости для квалифицированных кандидатов — важно не упускать талантливых специалистов 45+.
  *    *Философское обоснование*: Соответствует **утилитаристскому подходу** — максимизация полезности через найм действительно успешных кандидатов, независимо от возраста.
*    **Либертарианский подход:**
* *    *Почему уместен:* Фокусируется на защите индивидуальной автономии, свободе выбора и строгом соблюдении принципов меритократии. Он исключает любое принудительное перераспределение долей или искусственное групповое квотирование, которые нарушают право компании на свободный наем и права самих кандидатов на справедливую оценку их личных заслуг.
  *    *Философское обоснование:* Соответствует идеалу минимизации алгоритмического патернализма. Внедряемый механизм мониторинга и ручного аудита защищает свободу воли: алгоритм лишь предоставляет рекомендацию, а окончательное решение остается за человеком, который оценивает индивидуальные компетенции и достижения каждого кандидата вне зависимости от его принадлежности к какой-либо демографической группе.  
*    **Почему не «справедливость через неосведомлённость»**:
* *    Исключение признака `age` не устраняет дискриминацию, так как модель находит корреляты (прокси). Требуется активная коррекция смещений, а не пассивное игнорирование.

## Рекомендации по аудиту и пересмотру модели



| Мера | Периодичность | Ответственная сторона |
| :--- | :--- | :--- |
| **Мониторинг метрик справедливости**<br>• Демографический паритет ($\Delta \text{DP}$)<br>• Равные возможности ($\Delta \text{EO}$) | Ежеквартально | Команда этики ИИ «ТехноРекрут» |
| **Аудит прокси-признаков**<br>(выявление скрытой корреляции с возрастом кандидата) | При каждом обновлении модели | Data Science команда |
| **Переоценка весов признаков**<br>с применением методов снижения предвзятости *(debiasing)* | Раз в 6 месяцев | Команда ML-инженеров |
| **Внешний этический аудит** | Ежегодно | Независимая экспертная организация |
| **Анализ апелляций кандидатов** | Ежемесячно | Юридический отдел «МаркетПлюс» + Команда этики |
| **Публичная отчётность**<br>по зафиксированным показателям справедливости системы | Ежегодно | Пресс-служба «ТехноРекрут» |


### Технические рекомендации по смягчению смещений (Debiasing):

* **Pre-processing (подготовка данных):** Применить метод перевзвешивания (*reweighting*) исторических данных на этапе обучения, чтобы сбалансировать влияние возрастных групп без искусственного удаления кандидатов.
* **In-processing (изменение модели):** Внедрить состязательное снижение предвзятости (*adversarial debiasing*) или регуляризацию справедливости (*fairness constraints*). Это заставит модель игнорировать скрытые прокси-признаки (`outdated_vocab`, `graduation_year`) при формировании финального скора.
* **Post-processing (коррекция вывода):** Оптимизировать единый порог принятия решений для всей выборки на основе максимизации метрики *Equal Opportunity* (выравнивание True Positive Rate). 

  > ⚠️ **Важно:** Использование раздельных (дифференцированных) порогов для разных возрастов недопустимо, так как это нарушает запрет на прямую дискриминацию (ст. 3 ТК РФ). Порог должен быть строго один, но выбранный с учетом минимизации общего дисбаланса.
  >    
  > Оптимизация единого порога проводится на валидационной выборке с использованием метрики `Equalized Odds`, чтобы найти точку, в которой общая точность модели сохраняется, а разрыв в `TPR` между возрастными группами минимизируется в допустимых пределах. Это обеспечивает соблюдение требования единого стандарта оценки для всех кандидатов.
